## Using evolutionary algorithms to fine tune quantised FINN model on device

### Utils

There is a util for converting between npy and dat given the folding characteristics of the model.

Given the decoupled weights and the validation/test datasets, they can be viewed as 'unseen patient specific data to fine tune to'

### Questions

- How do I select which weights to put through Differential Evolution?
- Identify fraction of weights, DE param values, population size, end conditions, DE mutation and crossover method choice

### Tasks

- Run a PyTorch gradient descent loop on the ARM Cortex A53 on the ZCU-104 to set a baseline to compare against.
- Check out nevergrad library pre-implemented methods DiscreteDE, DiscreteLengler, DiscreteOnePlusOne, see what modifications are required.
- 

In [1]:
import sys
import platform
print(sys.executable)
print(sys.version)
print(platform.machine())

/usr/local/share/pynq-venv/bin/python3
3.10.4 (main, Apr  2 2022, 09:04:19) [GCC 11.2.0]
aarch64


In [1]:
!pip show nevergrad

Name: nevergrad
Version: 1.0.12
Summary: A Python toolbox for performing gradient-free optimization
Home-page: https://github.com/facebookresearch/nevergrad
Author: Facebook AI Research
Author-email: 
License: MIT
Location: /usr/local/share/pynq-venv/lib/python3.10/site-packages
Requires: bayesian-optimization, cma, directsearch, numpy, pandas, typing-extensions
Required-by: 


### Nevergrad differential evolution fine tuning loop with hardware eval and random mask

`ng.DifferentialEvolution` snapped to integer values



In [24]:
import sys 
sys.path.insert(0, "deploy/driver/")
import numpy as np
from driver_base import FINNExampleOverlay
from qonnx.core.datatype import DataType
from pynq.pl_server.device import Device
import qonnx
import os
from driver import io_shape_dict
import nevergrad as ng # library by facebook research implementing evolutionary and gradient free optimiser methods
from packing import pack_layer
import json

class evolution_tuner:
    # constr
    """
    Params:
    
    samples, labels: I/O "local data" to evaluate on
    starting_weight_dir: baseline weights path — contains both the .npy source
        weights and is used as the pack target for the untouched initial .dat
        weights (e.g. "runtime_weights_initial"). Never written to by the
        tuning loop itself, only by load_initial_weights.
    accelerator: FINNExampleOverlay accelerator instance with decoupled weights
    accel_info: folding factors for layer packing
    candidates: mask of weights up for modification
    runtime_weight_dir: working dir the accelerator loads from / the tuning
        loop overwrites every iteration (e.g. "runtime_weights")
    """
    def __init__(
        self,
        train_samples : str,
        train_labels : str,
        val_samples : str,
        val_labels : str,
        starting_weight_dir : str,
        accelerator : FINNExampleOverlay,
        accel_info : str = "final_hw_config.json", # path to accel final folding config file describing pe, simd needed for pack_layer
        candidates : np.ndarray = None,
        runtime_weight_dir : str = "deploy/driver/runtime_weights",
    ):
        self.train_samples = np.load(train_samples)
        self.train_labels = np.load(train_labels)
        self.train_size = self.train_samples.shape[0]
        
        self.val_samples = np.load(val_samples)
        self.val_labels = np.load(val_labels)
        self.val_size = self.val_samples.shape[0]
        
        self.accelerator = accelerator
        self.starting_weight_dir = starting_weight_dir
        self.starting_weights = [np.load(os.path.join(starting_weight_dir, f"{x}_0_StreamingDataflowPartition_{x}_MatrixVectorActivation_0.npy")) 
                                 for x in range(1, 5)]
        self.candidates = candidates
        self.runtime_weight_dir = runtime_weight_dir
        with open(accel_info, "r") as f:
            self.accel_info = json.load(f)
        
    
    # hardware valid method
    def pl_acc_eval(self, mode) -> float:
        # mode can be "train" or "val", sample selector
        if mode == "train":
            accel_out = self.accelerator.execute(self.train_samples)
            # compare ground truth labels with accelerator output
            result = np.where(self.train_labels == accel_out.flatten(), True, False)
            result_quant = np.unique(result, return_counts=True)
            correct_num = result_quant[1][1]
            test_acc = (correct_num / self.train_size)
            
        elif mode == "val":
            self.accelerator.batch_size = self.val_size
            accel_out = self.accelerator.execute(self.val_samples)
            # compare ground truth labels with accelerator output
            result = np.where(self.val_labels == accel_out.flatten(), True, False)
            self.accelerator.batch_size = self.train_size
            result_quant = np.unique(result, return_counts=True)
            correct_num = result_quant[1][1]
            test_acc = (correct_num / self.val_size)     
        
        return test_acc

    def load_initial_weights(self):
        """
        Pack starting_weights (loaded from starting_weight_dir at init time)
        into starting_weight_dir's .dat layout, then point the accelerator at
        runtime_weight_dir and load. Call this any time you need to reset the
        device back to its original state — starting_weight_dir is never
        touched by discrete_de_tuning_loop's per-iteration writes.
        """
        for l in range(1, len(self.starting_weights) + 1):
            np_layer = self.starting_weights[l - 1]
            dat_path = os.path.join(self.runtime_weight_dir, f"{l}_0_StreamingDataflowPartition_{l}_MatrixVectorActivation_0.dat")
            pe = self.accel_info[f"MatrixVectorActivation_{l - 1}"]["PE"]
            simd = self.accel_info[f"MatrixVectorActivation_{l - 1}"]["SIMD"]
            pack_layer(np_layer, dat_path, pe, simd, wdt="INT4")

        self.accelerator.runtime_weight_dir = self.runtime_weight_dir
        self.accelerator.load_runtime_weights()

    def discrete_de_tuning_loop(
        self, 
        layer_index : int = 1,
        budget : int = 100,
        patience : int = 100,
        bw : int = 4,
        verbose = True,
        mask_mode : str = "random",  # "random" or "column" or "row"
        p_cand : float = 0.3,        # fraction of weights (random), columns (column) or rows (row) to select
        cr = 0.5,
    ):
        I  = 2 ** (bw - 1) - 1    # = 7 boundary for bitwidth
        baseline  = self.starting_weights[layer_index] # iterate through the layers of the network
        flat_base = baseline.flatten()
        
        if self.candidates is None:
            rng = np.random.default_rng(seed=42)
            if mask_mode == "column":
                # select p_cand fraction of columns; all rows in chosen columns are candidates (originate from same activation)
                n_cols = baseline.shape[1]
                col_mask = rng.random(n_cols) < p_cand
                # repeat until at least one column is included in the mask
                while len(np.unique(col_mask)) == 1:
                    col_mask = rng.random(n_cols) < p_cand
                self.candidates = np.zeros(baseline.shape, dtype=bool)
                self.candidates[:, col_mask] = True
            elif mask_mode == "row":
                # select p_cand fraction of rows; all cols in chosen rows are candidates (map to the same output neuron)
                n_rows = baseline.shape[0]
                row_mask = rng.random(n_rows) < p_cand
                # repeat until at least one row is included in the mask
                while len(np.unique(row_mask)) == 1:
                    row_mask = rng.random(n_rows) < p_cand
                self.candidates = np.zeros(baseline.shape, dtype=bool)
                self.candidates[row_mask, :] = True
            elif mask_mode == "random":
                self.candidates = rng.random(baseline.shape) < p_cand
                # repeat until at least one weight is included in the mask
                while len(np.unique(self.candidates)) == 1:
                    self.candidates = rng.random(baseline.shape) < p_cand
            else:
                raise ValueError(f"unknown mask_mode '{mask_mode}', expected 'random', 'row' or 'column'")
                
        cand_flat = self.candidates.flatten()
        
        subset = flat_base[cand_flat] # filter down for candidate weights

        # ── Parametrisation ──────────────────────────────────────────────────────
        param = ng.p.Array(
            init=subset.astype(float),
            lower=-I,
            upper=I,
        ).set_integer_casting() # less overhead compared to Choice or TransitionChoice

        # ── Optimiser Selection ──────────────────────────────────────────────────
        # Preconfig DE instance that has crossover="dimension" set, so only one param is crossed over between samples
        discrete_de = ng.optimizers.DiscreteDE
        
        # Variable CR DE instance
        variable_cr_de = ng.families.DifferentialEvolution(crossover=cr)
        
        selected_de = variable_cr_de
        
        optimizer = selected_de(
            parametrization=param,
            budget=budget,
            num_workers=0,
        )

        # ── Baseline measurement ─────────────────────────────────────────────────
        # seed runtime_weight_dir from the untouched starting_weight_dir, then measure
        self.load_initial_weights()
        baseline_train_loss = 1.0 - self.pl_acc_eval("train")
        baseline_val_loss = 1.0 - self.pl_acc_eval("val")
        print(f"Baseline training set loss: {baseline_train_loss:.4f}  (acc: {1 - baseline_train_loss:.4f})")
        print(f"Baseline validation set loss: {baseline_val_loss:.4f}  (acc: {1 - baseline_val_loss:.4f})")
        
        best_val_loss = baseline_val_loss
        best_weights = baseline.copy()
        val_loss_history = [baseline_val_loss]
        evals_since_imp = 0

        # ── Ask/tell loop ────────────────────────────────────────────────────────
        # the nevergrad optimiser keeps track of the population, modifies based on candidate loss
        # new sample solution can be requested using optimiser.ask, loss can be fed in using optimiser.tell
        for i in range(budget):
            candidate = optimizer.ask()

            # Reconstruct full weight array
            full_weights = flat_base.copy()
            full_weights[cand_flat] = candidate.value
            full_weights_shaped = full_weights.reshape(baseline.shape).astype(np.int8)

            # existing pipeline — always written to runtime_weight_dir, never starting_weight_dir
            mvau_num = list(range(1, len(self.starting_weights) + 1))[layer_index]
            dat_path = os.path.join(self.runtime_weight_dir, f"{mvau_num}_0_StreamingDataflowPartition_{mvau_num}_MatrixVectorActivation_0.dat")
            pe = self.accel_info[f"MatrixVectorActivation_{mvau_num - 1}"]["PE"] # contrary to dat files, cfg has mvau indices 0, ..., L-1
            simd = self.accel_info[f"MatrixVectorActivation_{mvau_num - 1}"]["SIMD"]
            pack_layer(full_weights_shaped, dat_path, pe, simd, wdt="INT4")
            
            # load most recent weights into accelerator and run eval
            self.accelerator.load_runtime_weights()
            train_loss = 1.0 - self.pl_acc_eval("train")
            val_loss = 1.0 - self.pl_acc_eval("val")
            val_loss_history.append(val_loss)
            
            # feed results back into optimiser for modification
            optimizer.tell(candidate, train_loss) # we tell the optim the performance on the training set

            if val_loss <= best_val_loss:
                best_val_loss       = val_loss
                best_weights    = full_weights_shaped.copy()
                evals_since_imp = 0
            else:
                evals_since_imp += 1
                
            if verbose:
                print(f"iter {i:4d} | train loss {train_loss:.4f} | validation loss {val_loss:.4f} | best val. loss {best_val_loss:.4f}")

            if evals_since_imp >= patience:
                print(f"Stagnated at iter {i} — stopping")
                break

        # ── Save best weights ────────────────────────────────────────────────────
        mvau_num = list(range(1, len(self.starting_weights) + 1))[layer_index]
        dat_path = os.path.join(self.runtime_weight_dir, f"{mvau_num}_0_StreamingDataflowPartition_{mvau_num}_MatrixVectorActivation_0.dat")
        pe = self.accel_info[f"MatrixVectorActivation_{mvau_num - 1}"]["PE"]
        simd = self.accel_info[f"MatrixVectorActivation_{mvau_num - 1}"]["SIMD"]
        pack_layer(best_weights, dat_path, pe, simd, wdt="INT4")
        self.accelerator.load_runtime_weights()
        print(f"Best weights saved to: {dat_path}")

        return best_weights, best_val_loss, val_loss_history, self.candidates

In [25]:
import time
# select input-output data
# speech_samples_path = "data/google_speech/google_speech_X_test.npy"
# speech_labels_path = "data/google_speech/google_speech_y_test.npy"
# speech_samples = np.load(speech_samples_path)
# speech_labels = np.load(speech_labels_path)

train_samples = "split_data/train_set_X.npy"
train_labels = "split_data/train_set_y.npy"
val_samples = "split_data/val_set_X.npy"
val_labels = "split_data/val_set_y.npy"

# initialise accelerator and run baseline accuracy test
runtime_weight_dir = "deploy/driver/runtime_weights_initial"
bitfile = "bitfile/finn-accel.bit"
devID = 0
device = Device.devices[devID]
platform = "zynq-iodma"
num_input_samples = np.load(train_samples).shape[0]

accel = FINNExampleOverlay(
    bitfile_name = bitfile, platform = platform, 
    io_shape_dict = io_shape_dict, batch_size = num_input_samples,
    runtime_weight_dir = runtime_weight_dir, device = device
)

start_time = time.time()

# accuracy eval 
diff_tuner = evolution_tuner(train_samples, train_labels, val_samples, val_labels, runtime_weight_dir, accel) #, candidates=candidates)
# diff_tuner.load_initial_weights()
baseline_acc = diff_tuner.pl_acc_eval("val")
print(f"Baseline accuracy: {baseline_acc * 100:.2f}%")

# tuning loop
best_weights, best_loss, loss_history, candidates = diff_tuner.discrete_de_tuning_loop()

# run a val test at the end to get the best val acc
post_tune_acc = (1 - best_loss)

print(f"Post tuning accuracy: {post_tune_acc * 100:.2f}%")
end_time = time.time()
wall_time = end_time - start_time
print(f"Wall time: {wall_time:.3f}s")

Baseline accuracy: 77.46%
Baseline training set loss: 0.2244  (acc: 0.7756)
Baseline validation set loss: 0.2254  (acc: 0.7746)
iter    0 | train loss 0.2355 | validation loss 0.2398 | best val. loss 0.2254
iter    1 | train loss 0.2223 | validation loss 0.2049 | best val. loss 0.2049
iter    2 | train loss 0.2296 | validation loss 0.2152 | best val. loss 0.2049
iter    3 | train loss 0.2339 | validation loss 0.2193 | best val. loss 0.2049
iter    4 | train loss 0.2294 | validation loss 0.2213 | best val. loss 0.2049
iter    5 | train loss 0.2312 | validation loss 0.2254 | best val. loss 0.2049
iter    6 | train loss 0.2285 | validation loss 0.2193 | best val. loss 0.2049
iter    7 | train loss 0.2335 | validation loss 0.2131 | best val. loss 0.2049
iter    8 | train loss 0.2287 | validation loss 0.2111 | best val. loss 0.2049
iter    9 | train loss 0.2262 | validation loss 0.2213 | best val. loss 0.2049
iter   10 | train loss 0.2202 | validation loss 0.2254 | best val. loss 0.2049
ite

### Save latest run as .npy to checkpoint_weights to continue tuning from this stage

In [16]:
import time

start_time = time.time()

npy_dir = "deploy/driver/rows_weights"
file_name = f"4_0_StreamingDataflowPartition_4_MatrixVectorActivation_0.npy"
full_path = os.path.join(npy_dir, file_name)
source_dim = np.load(full_path).shape
if source_dim == best_weights.shape:
    np.save(full_path, best_weights)
    print(full_path)
else:
    print("dim not matching, check weight id")
    
end_time = time.time()
wall_time = end_time - start_time
print(f"Wall time: {wall_time}")

deploy/driver/rows_weights/4_0_StreamingDataflowPartition_4_MatrixVectorActivation_0.npy
Wall time: 0.005117654800415039


### Update .dat files in checkpoint_weights based on current .npy files in that directory (apply .npy modification)

In [17]:
# npy files at runtime_weights_initial were never overwritten
from packing import pack_layer
import numpy as np
import os
import json

with open("final_hw_config.json", "r") as f:
    cfg = json.load(f)

base_dir = "deploy/driver/rows_weights"

for i in range(1, 5):
    start_time = time.time()
    
    npy_name = f"{i}_0_StreamingDataflowPartition_{i}_MatrixVectorActivation_0.npy"
    npy_path = os.path.join(base_dir, npy_name)
    dat_name = npy_name[:-3] + "dat"
    dat_path = os.path.join(base_dir, dat_name)
    pe = cfg[f"MatrixVectorActivation_{i - 1}"]["PE"]
    simd = cfg[f"MatrixVectorActivation_{i - 1}"]["SIMD"]
    wdt = "INT4"
    pack_layer(npy_path, dat_path, pe, simd, wdt)
    
    end_time = time.time()
    wall_time = end_time - start_time
    print(f"Layer {i} wall time: {wall_time:.2f}s")

Layer 1 wall time: 1.89s
Layer 2 wall time: 0.98s
Layer 3 wall time: 0.98s
Layer 4 wall time: 0.08s


In [64]:
input_samples = np.load("data/google_speech/google_speech_y_test.npy")
input_class_counts = list(np.unique(input_samples, return_counts=True)[1])
print(input_class_counts)

[419, 405, 425, 406, 412, 396, 396, 402, 411, 402, 400, 400]


### Class specific accuracy statistics
It might help both the tuning process and accuracy if the last two classes, silence and unknown were removed as classes or were not tuned for in the validation set, but this is just a model-specific issue to consider

In [20]:
from collections import defaultdict
import math

label_dict = {0 : 'yes', 1 : 'no', 2 : 'up', 3 : 'down', 4 : 'left', 5 : 'right', 6 : 'on', 7 : 'off', 8 : 'stop', 
                9 : 'go', 10 : 'unknown', 11 : 'silence'}

input_samples = np.load("data/google_speech/google_speech_X_test.npy")
input_labels = np.load("data/google_speech/google_speech_y_test.npy")
input_class_counts = list(np.unique(input_labels, return_counts=True)[1])
print(input_class_counts)

# load weights at the runtime weight dir into the accelerator
accel.batch_size = input_samples.shape[0]
accel.runtime_weight_dir = "deploy/driver/rows_weights"
accel.load_runtime_weights()

errors = defaultdict(int)

# accuracy per class test
accel_out = accel.execute(input_samples).flatten()
print(list(np.unique(accel_out, return_counts=True)[1]))

for s in range(len(input_labels)):
    if accel_out[s] != input_labels[s]:
        errors[label_dict[input_labels[s]]] += 1
accs = []
        
for cl in range(len(input_class_counts)):
    label = label_dict[cl]
    num_inc = errors[label]
    num_tot = input_class_counts[cl]
    acc = (1 - num_inc / num_tot) * 100
    accs.append(acc)
    print(f"Errors for label '{label}': {num_inc}/{num_tot} = {acc:.2f}% accuracy")
    
print(f"Total: {sum([e for e in errors.values()])}/{sum(input_class_counts)}")

avg = sum(accs) / 12
std_dev = math.sqrt(sum([(a - avg) ** 2 for a in accs]) / 11)
print(f"stddev: {std_dev:.3f}")
    

[419, 405, 425, 406, 412, 396, 396, 402, 411, 402, 400, 400]
[454, 339, 370, 420, 415, 438, 406, 374, 373, 476, 407, 402]
Errors for label 'yes': 39/419 = 90.69% accuracy
Errors for label 'no': 151/405 = 62.72% accuracy
Errors for label 'up': 113/425 = 73.41% accuracy
Errors for label 'down': 100/406 = 75.37% accuracy
Errors for label 'left': 98/412 = 76.21% accuracy
Errors for label 'right': 35/396 = 91.16% accuracy
Errors for label 'on': 63/396 = 84.09% accuracy
Errors for label 'off': 83/402 = 79.35% accuracy
Errors for label 'stop': 72/411 = 82.48% accuracy
Errors for label 'go': 107/402 = 73.38% accuracy
Errors for label 'unknown': 7/400 = 98.25% accuracy
Errors for label 'silence': 171/400 = 57.25% accuracy
Total: 1039/4874
stddev: 11.733


In [21]:
from collections import defaultdict
import math

label_dict = {0 : 'yes', 1 : 'no', 2 : 'up', 3 : 'down', 4 : 'left', 5 : 'right', 6 : 'on', 7 : 'off', 8 : 'stop', 
                9 : 'go', 10 : 'unknown', 11 : 'silence'}

input_samples = np.load("data/google_speech/google_speech_X_test.npy")
input_labels = np.load("data/google_speech/google_speech_y_test.npy")
input_class_counts = list(np.unique(input_labels, return_counts=True)[1])
print(input_class_counts)

# load weights at the runtime weight dir into the accelerator
accel.runtime_weight_dir = "deploy/driver/runtime_weights_initial"
accel.load_runtime_weights()

errors = defaultdict(int)

# accuracy per class test
accel_out = accel.execute(input_samples).flatten()
print(list(np.unique(accel_out, return_counts=True)[1]))

for s in range(len(input_labels)):
    if accel_out[s] != input_labels[s]:
        errors[label_dict[input_labels[s]]] += 1
accs = []
        
for cl in range(len(input_class_counts)):
    label = label_dict[cl]
    num_inc = errors[label]
    num_tot = input_class_counts[cl]
    acc = (1 - num_inc / num_tot) * 100
    accs.append(acc)
    print(f"Errors for label '{label}': {num_inc}/{num_tot} = {acc:.2f}% accuracy")
    
print(f"Total: {sum([e for e in errors.values()])}/{sum(input_class_counts)}")

avg = sum(accs) / 12
std_dev = math.sqrt(sum([(a - avg) ** 2 for a in accs]) / 11)
print(f"stddev: {std_dev:.3f}")
    

[419, 405, 425, 406, 412, 396, 396, 402, 411, 402, 400, 400]
[463, 513, 359, 401, 427, 424, 428, 363, 352, 424, 414, 306]
Errors for label 'yes': 40/419 = 90.45% accuracy
Errors for label 'no': 93/405 = 77.04% accuracy
Errors for label 'up': 121/425 = 71.53% accuracy
Errors for label 'down': 112/406 = 72.41% accuracy
Errors for label 'left': 98/412 = 76.21% accuracy
Errors for label 'right': 43/396 = 89.14% accuracy
Errors for label 'on': 58/396 = 85.35% accuracy
Errors for label 'off': 92/402 = 77.11% accuracy
Errors for label 'stop': 85/411 = 79.32% accuracy
Errors for label 'go': 145/402 = 63.93% accuracy
Errors for label 'unknown': 5/400 = 98.75% accuracy
Errors for label 'silence': 202/400 = 49.50% accuracy
Total: 1094/4874
stddev: 12.974


In [4]:
train_samples = "split_data/train_set_X.npy"
val_samples = "split_data/val_set_X.npy"
print(np.load(train_samples).shape)
print(np.load(val_samples).shape)

(4386, 490)
(488, 490)
